# 01 — PAO Metadata Preparation

Prepares the raw PAO logbook Excel workbook (`Book1.xlsx`) into a clean, deduplicated, leak-free dataset saved to `data/processed/pao_metadata_v2.parquet`.

### Labeling Rules (`AOsMEMOs` vs `COMMUNICATION`)
- **`AOsMEMOs`**: Official Provincial Government of Cagayan (Governor / PA / internal PGC) Executive Orders, Administrative Orders, and Memorandum Orders / Memoranda.
- **`COMMUNICATION`**: Letters, replies, marginal notes, invitations/notices of meeting (even if referencing a draft order), Travel Orders, COA Audit Observation Memoranda (AOMs), and external national-agency advisories.
- **`needs_domain_review`**: Ambiguous cases (e.g., Special Orders, external agency AOs/EOs, dissemination notices) exported to `data/interim/pao_ask_domain_expert.csv` for PAO staff confirmation and excluded from baseline training until confirmed.


In [35]:
import re
import pandas as pd
from pathlib import Path

pd.set_option("display.max_colwidth", None)

RAW_FILE = Path("../data/raw/pao_metadata/Book1.xlsx")
REVIEW_FILE = Path("../data/interim/pao_label_review.csv")
ASK_FILE = Path("../data/interim/pao_ask_domain_expert.csv")
OUTPUT_FILE = Path("../data/processed/pao_metadata_v2.parquet")

print("raw file:   ", RAW_FILE.exists())
print("review file:", REVIEW_FILE.exists())
print("output file:", OUTPUT_FILE.exists())

raw file:    True
review file: True
output file: True


In [36]:
# Load the 4 logbook sheets with their true header rows
HEADER_ROWS = {
    "COMMUNICATION": 2,
    "CLEARANCE": 0,
    "AOsMEMOs": 2,
    "LEAVES": 0,
}

sheets = {}
for name, header in HEADER_ROWS.items():
    df = pd.read_excel(RAW_FILE, sheet_name=name, header=header)
    df.columns = df.columns.astype(str).str.strip()
    sheets[name] = df
    print(f"{name:15} rows={df.shape[0]:5} columns={df.shape[1]}")

COMMUNICATION   rows= 2285 columns=12
CLEARANCE       rows=  702 columns=16
AOsMEMOs        rows=   14 columns=6
LEAVES          rows=  657 columns=9


In [37]:
# Inspect columns and verify SUBJECT vs DESCRIPTION redundancy in COMMUNICATION
for name, df in sheets.items():
    print(f"\n--- {name} ---")
    print(df.columns.tolist())

comm = sheets["COMMUNICATION"]
subject = comm["SUBJECT"].fillna("").astype(str).str.strip().str.upper()
desc = comm["DESCRIPTION"].fillna("").astype(str).str.strip().str.upper()

same = subject == desc
subject_in_desc = [s != "" and s in d for s, d in zip(subject, desc)]
print(f"\nCOMMUNICATION rows:          {len(comm)}")
print(f"DESCRIPTION same as SUBJECT: {same.sum()}")
print(f"SUBJECT contained in DESC:   {sum(subject_in_desc)}")


--- COMMUNICATION ---
['DATE', 'CONTROL#', 'SUBJECT', 'DESCRIPTION', 'TYPE', 'OFFICE/PERSON FROM', 'OFFICE/UNIT CONCERNED', 'REMARKS', 'REMARKS 2', 'Unnamed: 9', 'Unnamed: 10', 'Unnamed: 11']

--- CLEARANCE ---
['DATE', 'CONTROL#', 'FULL N AME', 'OFFICE', 'PURPOSE', 'REMARKS', 'RELEASING DATE', 'C/O', 'REMARKS 2', 'Unnamed: 9', 'Unnamed: 10', 'Unnamed: 11', 'Unnamed: 12', 'Unnamed: 13', 'Unnamed: 14', 'Unnamed: 15']

--- AOsMEMOs ---
['DATE RECEIVED', 'DOCUMENT DATE', 'AO/MEMO/EO #', 'SUBJECT', 'OFFICE/UNIT/PERSON FROM', 'REMARKS']

--- LEAVES ---
['DATE FILED', 'TRANSMITAL DATE', 'CONTROL#', 'NAME', 'TYPE OF LEAVE', 'REMARKS', 'DAY/S APPLIED', 'DATE RECEIVED', 'STATUS']

COMMUNICATION rows:          2285
DESCRIPTION same as SUBJECT: 1604
SUBJECT contained in DESC:   1722


In [38]:
# Remove completely empty rows/columns and Excel Unnamed scratch columns
for name, df in sheets.items():
    before = df.shape
    df = df.dropna(how="all")
    df = df.dropna(axis=1, how="all")
    df = df.loc[:, ~df.columns.str.startswith("Unnamed")]
    sheets[name] = df
    print(f"{name:15} before={before} after={df.shape}")

COMMUNICATION   before=(2285, 12) after=(2285, 9)
CLEARANCE       before=(702, 16) after=(702, 9)
AOsMEMOs        before=(14, 6) after=(14, 6)
LEAVES          before=(657, 9) after=(653, 9)


In [39]:
# Assign permanent row_id BEFORE any cross-sheet deduplication so saved review IDs never shift
for name, df in sheets.items():
    df = df.reset_index(drop=True)
    df["row_id"] = name + "-" + df.index.astype(str)
    sheets[name] = df

display(sheets["COMMUNICATION"][["row_id", "SUBJECT"]].head(3))

,row_id,SUBJECT
0,COMMUNICATION-0,REQUEST TO RENDER OVERTIME
1,COMMUNICATION-1,PFC MEETING
2,COMMUNICATION-2,NOTICE OF ASSIGNMENT/ GRANT OF SECURITY INTEREST


In [40]:
# Step 1: Remove cross-sheet duplicates (documents logged in BOTH AOsMEMOs and COMMUNICATION)
# Why: AOsMEMOs only starts in Jan 2026 and references twin COMMUNICATION entries in REMARKS or SUBJECT.
# Dropping the COMMUNICATION twin prevents the same physical document from leaking across CV folds.
comm = sheets["COMMUNICATION"]
ao = sheets["AOsMEMOs"]

# 1a) Exact CONTROL# cross-references stored in AOsMEMOs.REMARKS
ao_refs = set(ao["REMARKS"].dropna().astype(str).str.strip()) - {""}
by_ctrl = comm["CONTROL#"].fillna("").astype(str).str.strip().isin(ao_refs)

# 1b) Subject prefix overlap within same year (catches twins where AOsMEMOs strips "AO NO. ..." prefix)
def _norm(s):
    return re.sub(r"\s+", " ", str(s).upper()).strip()

comm_year = pd.to_datetime(comm["DATE"], errors="coerce").dt.year
ao_years = set(pd.to_datetime(ao["DATE RECEIVED"], errors="coerce").dt.year.dropna().astype(int))
ao_prefixes = [_norm(x)[:55] for x in ao["SUBJECT"].dropna() if len(_norm(x)) >= 25]
comm_subj_norm = comm["SUBJECT"].fillna("").map(_norm)
comm_desc_norm = comm["DESCRIPTION"].fillna("").map(_norm)
by_subj = comm_year.isin(ao_years) & (
    comm_subj_norm.map(lambda s: any(p in s for p in ao_prefixes))
    | comm_desc_norm.map(lambda s: any(p in s for p in ao_prefixes))
)

dup_mask = by_ctrl | by_subj
print("Dropping COMMUNICATION rows already logged in AOsMEMOs:", dup_mask.sum())
display(comm.loc[dup_mask, ["row_id", "CONTROL#", "SUBJECT"]])

sheets["COMMUNICATION"] = comm.loc[~dup_mask].copy()

Dropping COMMUNICATION rows already logged in AOsMEMOs: 13


,row_id,CONTROL#,SUBJECT
1512,COMMUNICATION-1512,2026-01-26-COM-IN-04,EXECUTIVE ORDER NO. 01 - STREAMLINING AND RATIONALIZING BUREAUCRATIC PROCESSES THROUGH FUNCTIONAL GROUPING OF DEPT. AND OFFICES
1527,COMMUNICATION-1527,2026-01-29-COM-IN-01,AO NO. 02 S. 2026 - ORDER ADOPTING THE GUIDELINES ON THE USE AND OPERATION OF THE PGC CAPITOL SPORTS CENTER AND FITNESS GYM
1567,COMMUNICATION-1567,2026-02-05-COM-IN-05,"ADMINISTRATIVE ORDER NO. 1A, SERIES OF 2026"
1572,COMMUNICATION-1572,2026-02-05-COM-IN-10,"SUPPLEMENTAL ADMINISTRATIVE ORDER NO. 03, SERIES OF 2026"
1615,COMMUNICATION-1615,2026-02-16-COM-IN-03,"EXECUTIVE ORDER NO.06, SERIES OF 2026"
1695,COMMUNICATION-1695,2026-03-13-COM-IN-01,ADMINISTRATIVE ORDER NO. 9 S. 2026 RE INSTITUTING THE STANDARD RATES FOR MEALS & SNACKS DURING OFFICIAL MEETING ETC. OF PGC
1696,COMMUNICATION-1696,2026-03-13-COM-IN-02,"EXCUTIVE ORDER NO. 11 S. 2026 RE AN EXECUTIVE ORDER ADOPTING ENERGY CONSERVATION, FUEL MANAGEMENT, AND COST-EFFICIENCY MEASURES IN THE PROVINCIAL GOVERNMENT OF CAGAYAN IN COMPLIANCE WITH MEMORANDUM CIRCULAR NO. 114, S. 2026 AND IN RESPONSE TO GLOBAL FUEL"
1713,COMMUNICATION-1713,2026-03-17-COM-IN-03,"ADMINISTRATIVE ORDER NO. 10 S. 2026 RE AN ORDER PRESCRIBING COMPREHENSIVE POLICIES, GUIDELINES, AND PROCEDURES ON THE GRANT AND AVAILMENT OF LEAVES OF ABSENCE FOR ALL OFFICIALS AND EMPLOYEES OF THE PROVINCIAL GOVERNMENT OF CAGAYAN (PGC)"
1755,COMMUNICATION-1755,2026-03-30-COM-IN-02,"MEMORANDUM 34A: REQUEST FOR PERMANENT ALTERNATE REPRESENTATIVE OF THE PMSMEDC, SEND VIA EMAIL ON OR BEFORE MARCH 31, 2025"
1802,COMMUNICATION-1802,2026-04-14-COM-IN-01,"MEMORANDUM ORDER NO. 41-C, S. 2026: DIRECTIVE ON MANDATORY COMPLIANCE WITH REPUBLIC ACT. NO. 11032 (EASE OF DOING BUSINESS AND EFFICIENT GOVERNMENT SERVICE DELIVERY ACT OF 2018) AND IMPLEMENTATION OF THE 2026 CART ACTION PLAN"


In [41]:
# Step 2: Choose text and provenance columns per sheet (provenance is NEVER passed to the classifier)
TEXT_COLUMNS = {
    "COMMUNICATION": ["SUBJECT", "DESCRIPTION"],
    "CLEARANCE": ["PURPOSE"],
    "AOsMEMOs": ["SUBJECT"],
    "LEAVES": ["TYPE OF LEAVE", "REMARKS"],
}

DATE_COLUMNS = {
    "COMMUNICATION": "DATE",
    "CLEARANCE": "DATE",
    "AOsMEMOs": "DATE RECEIVED",
    "LEAVES": "DATE FILED",
}

CONTROL_COLUMNS = {
    "COMMUNICATION": "CONTROL#",
    "CLEARANCE": "CONTROL#",
    "AOsMEMOs": "AO/MEMO/EO #",
    "LEAVES": "CONTROL#",
}

for name, cols in TEXT_COLUMNS.items():
    missing = [c for c in cols if c not in sheets[name].columns]
    assert not missing, f"{name} missing {missing}"
    print(f"{name:15} text={cols} | date={DATE_COLUMNS[name]} | control={CONTROL_COLUMNS[name]}")

COMMUNICATION   text=['SUBJECT', 'DESCRIPTION'] | date=DATE | control=CONTROL#
CLEARANCE       text=['PURPOSE'] | date=DATE | control=CONTROL#
AOsMEMOs        text=['SUBJECT'] | date=DATE RECEIVED | control=AO/MEMO/EO #
LEAVES          text=['TYPE OF LEAVE', 'REMARKS'] | date=DATE FILED | control=CONTROL#


In [42]:
# Build unified datasets table without duplicated SUBJECT+DESCRIPTION text
def join_unique(values):
    result = ""
    for value in values:
        value = str(value).strip()
        if value == "" or value.lower() == "nan":
            continue
        if value.upper() in result.upper():
            continue
        if result.upper() in value.upper():
            result = value
        else:
            result = (result + " " + value).strip()
    return result

parts = []
for name, cols in TEXT_COLUMNS.items():
    df = sheets[name]
    text = df[cols].fillna("").apply(join_unique, axis=1)
    parts.append(pd.DataFrame({
        "row_id": df["row_id"],
        "source_sheet": name,
        "date": pd.to_datetime(df[DATE_COLUMNS[name]], errors="coerce").dt.strftime("%Y-%m-%d").fillna(""),
        "control_no": df[CONTROL_COLUMNS[name]].fillna("").astype(str).str.strip(),
        "document_text": text,
        "label": name,
    }))

datasets = pd.concat(parts, ignore_index=True)
print("Combined shape:", datasets.shape)
print(datasets["label"].value_counts())

Combined shape: (3641, 6)
label
COMMUNICATION    2272
CLEARANCE         702
LEAVES            653
AOsMEMOs           14
Name: count, dtype: int64


In [43]:
# Normalize whitespace/case and remove rows with blank document_text
def clean_text(text):
    text = str(text).upper()
    text = re.sub(r"\s+", " ", text)
    return text.strip()

datasets["document_text"] = datasets["document_text"].map(clean_text)

blank = datasets["document_text"].eq("")
print("Blank rows removed:", blank.sum())
datasets = datasets[~blank].reset_index(drop=True)

# Verify no doubled sentences remain in COMMUNICATION
comm_text = datasets.loc[datasets["label"] == "COMMUNICATION", "document_text"]
has_repeat = lambda t: len(t) > 20 and t[:len(t) // 2].strip() == t[len(t) // 2:].strip()
print("COMMUNICATION texts repeated twice:", comm_text.map(has_repeat).sum())
print("Rows after removing blanks:", len(datasets))
print(datasets["label"].value_counts())

Blank rows removed: 10
COMMUNICATION texts repeated twice: 5
Rows after removing blanks: 3631
label
COMMUNICATION    2272
CLEARANCE         693
LEAVES            652
AOsMEMOs           14
Name: count, dtype: int64


In [44]:
# Step 3: Find COMMUNICATION rows that look like orders/memos (v2 expanded regex + COMMUNICATION TYPE column)
# Catches punctuation/abbreviation variants found in v1 error analysis (MEMO. ORDER, MEMO NO., ADMIN ORDER, ^MEMORANDUM, ^MEMO)
ORDER_PATTERN = (
    r"\b(?:ADMINISTRATIVE ORDER|ADMIN\.?\s+ORDER|EXECUTIVE ORDER|EXCUTIVE ORDER|"
    r"MEMORANDUM ORDER|MEMO\.?\s+ORDER|SPECIAL ORDER|"
    r"MEMORANDUM\s+NO\.?|MEMO\.?\s+NO\.?|MEMORANDUM\s+\d+|MEMO\.?\s+\d+|"
    r"AO\s+NO\.?|EO\s+NO\.?)"
    r"|^MEMORANDUM\b|^MEMO\b"
)

comm_type_map = (
    sheets["COMMUNICATION"]
    .set_index("row_id")["TYPE"]
    .fillna("")
    .astype(str)
    .str.strip()
    .str.upper()
)

is_order_text = datasets["document_text"].str.contains(ORDER_PATTERN, regex=True)
is_order_type = datasets["row_id"].map(comm_type_map).isin(["MEMO", "EO"])
candidates = datasets[(datasets["label"] == "COMMUNICATION") & (is_order_text | is_order_type)]

print("Candidate rows for label review:", len(candidates))

Candidate rows for label review: 180


In [45]:
# Load saved review table, drop removed cross-sheet duplicates, and append any newly found candidates
if REVIEW_FILE.exists():
    review = pd.read_csv(REVIEW_FILE, keep_default_na=False).set_index("row_id")
    review = review.loc[review.index.intersection(datasets["row_id"])].copy()
    new_ids = candidates["row_id"][~candidates["row_id"].isin(review.index)]
    if len(new_ids) > 0:
        new_rows = (
            candidates[candidates["row_id"].isin(new_ids)][["row_id", "document_text", "label"]]
            .rename(columns={"label": "original_label"})
            .set_index("row_id")
        )
        new_rows["reviewed_label"] = ""
        new_rows["review_status"] = "pending"
        review = pd.concat([review, new_rows])
        review.to_csv(REVIEW_FILE)
        print(f"Appended {len(new_rows)} new candidate rows")
else:
    review = (
        candidates[["row_id", "document_text", "label"]]
        .rename(columns={"label": "original_label"})
        .set_index("row_id")
    )
    review["reviewed_label"] = ""
    review["review_status"] = "pending"
    review.to_csv(REVIEW_FILE)

print("Review status counts:")
print(review["review_status"].value_counts())
print("\nReviewed label counts:")
print(review["reviewed_label"].value_counts())

Review status counts:
review_status
verified               167
needs_domain_review     13
Name: count, dtype: int64

Reviewed label counts:
reviewed_label
AOsMEMOs         148
COMMUNICATION     19
                  13
Name: count, dtype: int64


In [46]:
# Helper to mark review rows and persist immediately to CSV
def mark(row_ids, label=None, status="verified"):
    if status == "verified" and label is None:
        raise ValueError("verified rows need a label")
    review.loc[row_ids, "review_status"] = status
    if label is not None:
        review.loc[row_ids, "reviewed_label"] = label
    review.to_csv(REVIEW_FILE)
    print(review["review_status"].value_counts().to_dict())

# Export needs_domain_review rows for PAO staff confirmation
ask_pao = review[review["review_status"] == "needs_domain_review"]
ask_pao[["document_text", "original_label"]].to_csv(ASK_FILE)
print(f"{len(ask_pao)} rows saved to {ASK_FILE}")
display(ask_pao[["document_text"]])

13 rows saved to ..\data\interim\pao_ask_domain_expert.csv


,document_text
row_id,
COMMUNICATION-46,"SPECIAL ORDER NO.2 S. 2024 DESIGNATION AS IN-CHARGE, OFFICE OF THE PROVINCIAL ADMINISTRATION"
COMMUNICATION-190,"INTERNAL AUDIT MEMORANDUM NO. 01S, 2024 REITERATION OF MEMORANDUM DATED 12/22/2024 ISSUED BY PEO-MOTORPOOL DIVISION RE: GUIDELINES FOR MOTOR VEHICLE REPAIR AND MAINTENANCE"
COMMUNICATION-467,"DOST AO NO. 011, SERIES OF 2020 RE: REVISED GUIDELINES FOR GRANTS-IN-AID PROGRAM FOR DEPARTMENT OF SCIENCE & TECHNOLOGY AND ITS AGENCIES FOR THE LIQUIDATION."
COMMUNICATION-527,"SPECIAL ORDER NO. 04 S.2025; ENGR. ALICE EMMA A. PASON THE PROVINCIAL ASSESSOR OF THIS PROVINCE DESIGNATED AS OFFICER-IN-CHARGE, PROVINCIAL ADMINISTRATOR'S OFFICE WHILE THE PROVINCIAL ADMINISTARTOR ATTY. MARIA ROSARIO S. MAMBA-VILLAFLOR I SON OFFICIAL BUSINESS IN METRO MANILA"
COMMUNICATION-802,"SPECIAL ORDER NO. 10, S. 2025: DESIGNATION AS OIC, PROVINCIAL ADMINISTRATOR'S OFFICE OF ENGR. ALICE EMMA A. PASON"
COMMUNICATION-919,"SPECIAL ORDER-01-S.2025 SO-01-S.2025 - DESIGNATION AS OFFICER-IN-CHARGE, PROVINCIAL ADMINISTRATOR - ENGR. ALICE EMMA A. PASON FROM JULY 14-AUGUST 1, 2025"
COMMUNICATION-1008,"MEMORANDUM NO. 013 S. 2025 PRIORITIZATION OF SGLG-RELATED PROGRAMS IN ANNUAL OPERATIONS PLAN, (AOP) AND ANNUAL INVESTMENT PLAN (AIP) 2026"
COMMUNICATION-1009,MEMORANDUM NO. 012 S. 2025 CREATION OF SGLG TECHNICAL WORKING GROUP
COMMUNICATION-1014,"DESIGNATION OF MS. MARICAR LARDIZABAL AS OIC OF BOOKKEEPING DIVISION AND MS. MELISSA PEREZ TRANSMITTED COPY OF HRMO AS PER MEMO. ORDER NO. 78, 2025"


In [47]:
# Step 4: Apply verified labels and preserve review_status on every row
# Why: Keeping review_status prevents needs_domain_review rows from silently being trained as COMMUNICATION.
assert (review["review_status"] == "pending").sum() == 0, "Finish all pending review rows first"

datasets["original_label"] = datasets["label"]
datasets["review_status"] = "unreviewed_default"

datasets = datasets.set_index("row_id")
datasets.loc[review.index, "review_status"] = review["review_status"]

verified = review[review["review_status"] == "verified"]
datasets.loc[verified.index, "label"] = verified["reviewed_label"]
datasets = datasets.reset_index()

print("Labels changed:", (datasets["label"] != datasets["original_label"]).sum())
print("\nReview status counts:")
print(datasets["review_status"].value_counts())
print("\nOriginal vs reviewed label crosstab:")
display(pd.crosstab(datasets["original_label"], datasets["label"]))
print("\nFinal label counts (all rows):")
print(datasets["label"].value_counts())

Labels changed: 148

Review status counts:
review_status
unreviewed_default     3451
verified                167
needs_domain_review      13
Name: count, dtype: int64

Original vs reviewed label crosstab:


label,AOsMEMOs,CLEARANCE,COMMUNICATION,LEAVES
original_label,,,,
AOsMEMOs,14,0,0,0
CLEARANCE,0,693,0,0
COMMUNICATION,148,0,2124,0
LEAVES,0,0,0,652



Final label counts (all rows):
label
COMMUNICATION    2124
CLEARANCE         693
LEAVES            652
AOsMEMOs          162
Name: count, dtype: int64


In [48]:
# Step 5: Build normalized group_id (for leak-free CV) and flag CLEARANCE vs LEAVES text conflicts
def make_group_id(text):
    text = re.sub(r"[^A-Z0-9\s]", " ", str(text).upper())
    return re.sub(r"\s+", " ", text).strip()

datasets["group_id"] = datasets["document_text"].map(make_group_id)

clean_mask = datasets["review_status"] != "needs_domain_review"
labels_per_group = datasets.loc[clean_mask].groupby("group_id")["label"].nunique()
conflicting_groups = set(labels_per_group[labels_per_group > 1].index)

datasets["has_label_conflict"] = datasets["group_id"].isin(conflicting_groups)

print("Conflicting unique groups:", len(conflicting_groups))
print("Rows flagged has_label_conflict:", datasets["has_label_conflict"].sum())
display(
    datasets[datasets["has_label_conflict"]]
    .groupby(["group_id", "label"])
    .size()
    .unstack(fill_value=0)
)

Conflicting unique groups: 5
Rows flagged has_label_conflict: 354


label,CLEARANCE,LEAVES
group_id,,
MATERNITY,16,1
SICK LEAVE,20,186
STUDY LEAVE,11,1
TERMINAL,94,1
VACATION LEAVE,7,17


In [49]:
# Step 6: Export clean dataset to data/processed/pao_metadata_v2.parquet and verify
OUTPUT_FILE.parent.mkdir(parents=True, exist_ok=True)
assert "original_label" in datasets.columns, "Apply verified labels first"

cols_to_save = [
    "row_id",
    "source_sheet",
    "date",
    "control_no",
    "document_text",
    "label",
    "original_label",
    "review_status",
    "group_id",
    "has_label_conflict",
]

final = datasets[cols_to_save]
final.to_parquet(OUTPUT_FILE, index=False)

check = pd.read_parquet(OUTPUT_FILE)
print("Saved parquet:", OUTPUT_FILE, "| shape:", check.shape)
print("\nLabel counts (excluding needs_domain_review):")
print(check.loc[check["review_status"] != "needs_domain_review", "label"].value_counts())

Saved parquet: ..\data\processed\pao_metadata_v2.parquet | shape: (3631, 10)

Label counts (excluding needs_domain_review):
label
COMMUNICATION    2111
CLEARANCE         693
LEAVES            652
AOsMEMOs          162
Name: count, dtype: int64
